# 05 · Detector de bola no vídeo (Colab com GPU)

Pergunta: um detector pré-treinado (YOLO, classe COCO "sports ball") encontra a bola nos frames em
que o tracking da organização a perdeu?

- **Controle:** frames em que o tracker viu a bola. Medimos se o detector a encontra no mesmo lugar
  (posição do tracker projetada no vídeo pela homografia). Diz se o detector funciona nesta câmera.
- **Buracos:** frames sem bola no tracking. Medimos em quantos o detector acha algo dentro do campo.

Sem anotação manual e sem gabarito. O código é o mesmo de `src/bola/detector.py`, que também roda
localmente em CPU com uma amostra.

**Antes de rodar:** Ambiente de execução → Alterar o tipo → **GPU T4**. Os dados ficam no seu
Drive em `MyDrive/mtah-data/tactical/` (mesmo caminho do notebook 04). Nenhum dado é enviado a
serviços de IA; não use o assistente do Colab com este notebook.

In [ ]:
# Repositório e dependências
import os, subprocess, sys
REPO = '/content/more-than-a-hack-2027'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/Thiagouh/more-than-a-hack-2027.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'scikit-learn'], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
import torch
print('GPU disponível:', torch.cuda.is_available())

In [ ]:
# Dados no Drive (peça de autorização aparece aqui: aceite todas as permissões)
from google.colab import drive
drive.mount('/content/drive')
os.environ['MTAH_DATA_DIR'] = '/content/drive/MyDrive/mtah-data'
print(sorted(os.listdir(os.environ['MTAH_DATA_DIR'] + '/tactical')))

## Rodar o experimento

`--passo 1` usa todos os frames do trecho público (~6.600 de controle a cada 7 e ~1.000 de
buraco). `--modelo yolo11x.pt` é o maior e mais preciso; numa T4 leva poucos minutos.

In [ ]:
!python -m src.bola.detector --passo 1 --modelo yolo11x.pt --imgsz 1920

## Como ler o resultado

- Se o **controle** ficar baixo (o detector não acha a bola nem quando o tracker acha), um detector
  genérico não serve nesta câmera; o próximo passo seria ajuste fino com rótulos automáticos
  (posições do tracker projetadas no vídeo).
- Se o controle for alto e os **buracos** também mostrarem detecções dentro do campo, vale rodar no
  jogo inteiro e usar as detecções para preencher os buracos.

O arquivo de detecções fica em `outputs/bola/detector_yolo11x.json`; baixe-o pelo painel de
arquivos se quiser analisar localmente.